# 🔧 Lesson 14: Capstone Core — Code Sandbox + Voice Input

Building the heart of the Interview Copilot: live code execution, voice input, and the full session loop.

In [ ]:
import os, json, io, sys, time, subprocess, contextlib
from pathlib import Path
from openai import AzureOpenAI
from autogen import AssistantAgent, UserProxyAgent
from dotenv import load_dotenv
from rich.console import Console
from rich.syntax import Syntax
from rich.panel import Panel
load_dotenv()
console = Console()
client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION')
)
console.print('[bold green]🚀 Copilot Core — Ready![/bold green]')

## 🐍 Part 1: Secure Code Execution Sandbox

In [ ]:
def run_code_safely(code: str, test_input: str = None, timeout: int = 10) -> dict:
    """
    Execute Python code in a sandboxed subprocess.
    Returns: {success, output, error, runtime_ms}
    """
    start = time.time()
    
    try:
        result = subprocess.run(
            ['python', '-c', code],
            capture_output=True,
            text=True,
            timeout=timeout
        )
        runtime_ms = int((time.time() - start) * 1000)
        
        return {
            'success': result.returncode == 0,
            'output': result.stdout.strip(),
            'error': result.stderr.strip(),
            'runtime_ms': runtime_ms
        }
    except subprocess.TimeoutExpired:
        return {'success': False, 'output': '', 'error': 'TIMEOUT: Code ran too long!', 'runtime_ms': timeout*1000}
    except Exception as e:
        return {'success': False, 'output': '', 'error': str(e), 'runtime_ms': 0}


def run_with_test_cases(solution_code: str, test_cases: list) -> dict:
    """Run solution against multiple test cases."""
    results = []
    
    for i, tc in enumerate(test_cases):
        test_runner = solution_code + f"""

# Test case {i+1}
result = {tc['call']}
expected = {tc['expected']}
print(f'Test {i+1}: {{"PASS" if result == expected else "FAIL"}} | Got: {{result}} | Expected: {{expected}}')
"""
        result = run_code_safely(test_runner)
        results.append({'test': i+1, **result})
    
    passed = sum(1 for r in results if r['success'] and 'PASS' in r['output'])
    return {
        'passed': passed,
        'total': len(test_cases),
        'score': passed / len(test_cases),
        'results': results
    }


# Test the sandbox with Two Sum solution
two_sum_solution = """
def twoSum(nums, target):
    seen = {}
    for i, n in enumerate(nums):
        if target-n in seen:
            return [seen[target-n], i]
        seen[n] = i
"""

test_cases = [
    {'call': 'twoSum([2,7,11,15], 9)', 'expected': '[0,1]'},
    {'call': 'twoSum([3,2,4], 6)', 'expected': '[1,2]'},
    {'call': 'twoSum([3,3], 6)', 'expected': '[0,1]'},
]

sandbox_results = run_with_test_cases(two_sum_solution, test_cases)
console.print(Panel(
    f"Score: {sandbox_results['passed']}/{sandbox_results['total']} ({sandbox_results['score']*100:.0f}%)",
    title='🧪 Code Sandbox Results'
))
for r in sandbox_results['results']:
    console.print(f"  Test {r['test']}: {r['output']}")

## 🎤 Part 2: Voice Input with Azure Whisper

In [ ]:
# Voice transcription using Azure Whisper
def transcribe_voice(audio_file_path: str) -> str:
    """
    Transcribe audio file to text using Azure Whisper.
    Supports: mp3, mp4, mpeg, mpga, m4a, wav, webm
    """
    try:
        with open(audio_file_path, 'rb') as audio_file:
            transcription = client.audio.transcriptions.create(
                model=os.getenv('AZURE_WHISPER_DEPLOYMENT', 'whisper'),
                file=audio_file,
                language='en'
            )
        return transcription.text
    except Exception as e:
        return f'[Voice transcription error: {e}]'


def get_user_input(allow_voice: bool = True) -> dict:
    """
    Get input from user — text or voice.
    In production, this connects to a microphone stream.
    For demo, simulates voice with a pre-recorded file.
    """
    mode = input('Input mode? [t]ext / [v]oice: ').strip().lower()
    
    if mode == 'v' and allow_voice:
        audio_path = input('Audio file path (wav/mp3): ').strip()
        text = transcribe_voice(audio_path)
        console.print(f'[cyan]🎤 Transcribed:[/cyan] {text}')
        return {'mode': 'voice', 'text': text}
    else:
        text = input('Your answer: ')
        return {'mode': 'text', 'text': text}

console.print('[green]✅ Voice input handler ready[/green]')
console.print('[yellow]💡 Tip: In production, stream audio from microphone in real-time![/yellow]')

## 🤖 Part 3: Full Interview Session Loop

In [ ]:
import uuid
from dataclasses import dataclass, field
from typing import List

@dataclass
class InterviewSession:
    user_name: str
    problem_title: str
    session_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    hints_used: int = 0
    attempts: List[dict] = field(default_factory=list)
    start_time: float = field(default_factory=time.time)
    
    def add_attempt(self, code: str, test_results: dict):
        self.attempts.append({
            'attempt_num': len(self.attempts) + 1,
            'code_length': len(code),
            'score': test_results['score'],
            'passed': test_results['passed'],
            'total': test_results['total'],
            'time_elapsed': int(time.time() - self.start_time)
        })
    
    def final_score(self) -> float:
        if not self.attempts: return 0.0
        best = max(a['score'] for a in self.attempts)
        hint_penalty = self.hints_used * 0.1  # -10% per hint used
        return max(0.0, best - hint_penalty)
    
    def save(self):
        path = f'./copilot_data/session_{self.session_id}.json'
        data = {
            'session_id': self.session_id,
            'user': self.user_name,
            'problem': self.problem_title,
            'hints_used': self.hints_used,
            'final_score': self.final_score(),
            'attempts': self.attempts,
            'duration_seconds': int(time.time() - self.start_time)
        }
        Path('./copilot_data').mkdir(exist_ok=True)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        return path


def run_demo_session(problem_title='Two Sum', user_name='Avnish'):
    """Simulated interview session for demonstration."""
    session = InterviewSession(user_name=user_name, problem_title=problem_title)
    
    console.print(Panel(
        f'[bold]Welcome, {user_name}![/bold]\nProblem: {problem_title}\nSession: {session.session_id}',
        title='🎯 Interview Starting'
    ))
    
    # Simulate student solution
    student_code = """
def twoSum(nums, target):
    seen = {}
    for i, n in enumerate(nums):
        if target-n in seen:
            return [seen[target-n], i]
        seen[n] = i
"""
    
    # Display submitted code
    syntax = Syntax(student_code, 'python', theme='monokai')
    console.print(Panel(syntax, title='📝 Submitted Code'))
    
    # Run test cases
    test_cases = [
        {'call': 'twoSum([2,7,11,15], 9)', 'expected': '[0,1]'},
        {'call': 'twoSum([3,2,4], 6)', 'expected': '[1,2]'},
        {'call': 'twoSum([3,3], 6)', 'expected': '[0,1]'},
    ]
    
    results = run_with_test_cases(student_code, test_cases)
    session.add_attempt(student_code, results)
    
    # Display results
    for r in results['results']:
        console.print(f"  {'✅' if 'PASS' in r['output'] else '❌'} {r['output']}")
    
    final = session.final_score()
    path = session.save()
    
    console.print(Panel(
        f"Tests: {results['passed']}/{results['total']}\n"
        f"Hints used: {session.hints_used}\n"
        f"Final Score: {final*100:.0f}%\n"
        f"Session saved: {path}",
        title='🏆 Session Complete'
    ))
    return session

session = run_demo_session()

## ✅ Core Complete!
Lesson 15 adds the Streamlit dashboard, ChromaDB memory, and full production deployment.